# Traits

Welcome! Traits are Rust's way of defining shared behavior — like interfaces in other languages, but more powerful. A trait describes what a type *can do*: "this type can be printed," "this type can be compared," "this type can be iterated over." Traits enable polymorphism and code reuse while keeping everything type-safe.

## What Are Traits and Why Do They Matter?

Traits let you:
- Define shared behavior across different types
- Write generic code that works with any type implementing a trait
- Use the `derive` macro to auto-implement common traits

Think of traits as contracts: "if you implement this trait, you promise to provide these methods."

## Defining a Trait

Use the `trait` keyword to define a trait. Methods can have default implementations or just signatures (which implementors must provide).

In [ ]:
trait Greet {
    fn greet(&self) -> String;
}

struct Person { name: String }
struct Robot { id: u32 }

impl Greet for Person {
    fn greet(&self) -> String {
        format!("Hello, I'm {}", self.name)
    }
}

impl Greet for Robot {
    fn greet(&self) -> String {
        format!("Beep. I am robot #{}", self.id)
    }
}

let p = Person { name: String::from("Alice") };
let r = Robot { id: 42 };
println!("{}", p.greet());
println!("{}", r.greet());

## Default Methods

Traits can provide default implementations. Implementors can override them or use the default.

In [ ]:
trait Describable {
    fn describe(&self) -> String {
        String::from("(no description)")
    }
}

struct Item { name: String }

impl Describable for Item {
    fn describe(&self) -> String {
        format!("Item: {}", self.name)
    }
}

struct Mystery;
impl Describable for Mystery {}  // Uses default!

let item = Item { name: String::from("Sword") };
let mystery = Mystery;
println!("{}", item.describe());
println!("{}", mystery.describe());

## Trait Bounds: T: Display

When writing generic code, you can restrict `T` to types that implement a trait using `T: TraitName`. This lets you call trait methods on `T`.

In [ ]:
use std::fmt::Display;

fn print_twice<T: Display>(t: T) {
    println!("{}", t);
    println!("{}", t);
}

print_twice(42);
print_twice("hello");

## Where Clauses

When you have many trait bounds, `where` clauses keep the function signature readable.

In [ ]:
use std::fmt::Debug;

fn compare<T, U>(a: T, b: U)
where
    T: Display + Debug,
    U: Display + Debug,
{
    println!("a: {:?}, b: {:?}", a, b);
}

compare(1, "two");

## impl Trait: Argument and Return

`impl Trait` lets you say "some type that implements this trait" without naming the type. Use it for return types when you want to hide the concrete type, or for parameters when you want flexibility.

In [ ]:
fn greet_something(g: impl Greet) {
    println!("{}", g.greet());
}

fn get_greeter(is_human: bool) -> Box<dyn Greet> {
    if is_human {
        Box::new(Person { name: String::from("Bob") })
    } else {
        Box::new(Robot { id: 99 })
    }
}

greet_something(get_greeter(true));

## Supertraits

A trait can require that implementors also implement another trait. Use `trait Foo: Bar` — "to implement Foo, you must implement Bar first."

In [ ]:
trait Named {
    fn name(&self) -> &str;
}

trait NamedGreet: Named + Greet {
    fn named_greet(&self) -> String {
        format!("{} says: {}", self.name(), self.greet())
    }
}

impl Named for Person {
    fn name(&self) -> &str { &self.name }
}

impl NamedGreet for Person {}

let p = Person { name: String::from("Carol") };
println!("{}", p.named_greet());

## Common Standard Library Traits

| Trait | Purpose |
|-------|--------|
| `Debug` | `{:?}` formatting |
| `Display` | `{}` user-facing formatting |
| `Clone` | `.clone()` for explicit copies |
| `Copy` | Implicit copy (must also be Clone) |
| `PartialEq` / `Eq` | `==` and `!=` |
| `PartialOrd` / `Ord` | `<`, `>`, ordering |
| `Hash` | Hashing for HashMap keys |
| `Default` | `T::default()` |
| `From` / `Into` | Type conversion |
| `Iterator` | Iteration over collections |

In [ ]:
#[derive(Debug, Clone, PartialEq)]
struct Point { x: i32, y: i32 }

let p1 = Point { x: 1, y: 2 };
let p2 = p1.clone();
println!("{:?}", p1);
println!("p1 == p2: {}", p1 == p2);

## The Derive Macro

`#[derive(Trait)]` auto-implements common traits. The compiler generates the implementation for you. Available for: `Debug`, `Clone`, `Copy`, `PartialEq`, `Eq`, `PartialOrd`, `Ord`, `Hash`, `Default`.

In [ ]:
#[derive(Debug, Default)]
struct Config {
    timeout: u32,
    retries: u32,
}

let config = Config::default();
println!("{:?}", config);

## The Orphan Rule

You can only implement a trait for a type if either the trait or the type is defined in your crate. This prevents breaking others' code by implementing their traits for their types. It keeps the ecosystem stable.

## Trait Objects (dyn Trait) vs Generics

- **Generics** (`fn foo<T: Trait>(t: T)`): Compile-time polymorphism. Each concrete type gets its own code. No runtime cost.
- **Trait objects** (`fn foo(t: &dyn Trait)`): Runtime polymorphism. One function handles any implementor via a vtable. Slight runtime cost, but allows heterogeneous collections.

Use generics when you know the type at compile time. Use `dyn Trait` when you need to store different types in the same collection or pass "any implementor" at runtime.

In [ ]:
fn greet_all(items: &[&dyn Greet]) {
    for item in items {
        println!("{}", item.greet());
    }
}

let p = Person { name: String::from("Dave") };
let r = Robot { id: 1 };
let greeters: &[&dyn Greet] = &[&p, &r];
greet_all(greeters);

## When to Use Each

- **Generics + trait bounds**: Default choice. Zero-cost abstraction, compiler optimizes.
- **impl Trait**: Simpler syntax when you don't need to name the type. Good for return types.
- **dyn Trait**: When you need a collection of different types (e.g., `Vec<Box<dyn Drawable>`) or dynamic dispatch.

## Common Mistakes Beginners Make

1. **Forgetting to import the trait** — you need `use std::fmt::Display` (or similar) to use trait methods
2. **Confusing `impl Trait` and `dyn Trait`** — `impl` is static dispatch; `dyn` is dynamic
3. **Trait objects and `Sized`** — trait objects are unsized; use `&dyn Trait` or `Box<dyn Trait>`
4. **Over-deriving** — only derive traits you actually need

## Key Rules to Remember

- Traits define shared behavior; implement them with `impl Trait for Type`
- Use trait bounds (`T: Trait`) to constrain generics
- `impl Trait` = "some type implementing this trait" (static dispatch)
- `dyn Trait` = trait object for runtime polymorphism
- `#[derive(...)]` auto-implements common traits
- The orphan rule: implement a trait only if you own the trait or the type